In [ ]:
!pip install -U langchain langchain-community langchain-core
!pip install sentence-transformers faiss-cpu
!pip install transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import json

print(os.listdir('/content/drive/MyDrive/hotpot'))

train_path = '/content/drive/MyDrive/hotpot/train-v2.0.json'
dev_path = '/content/drive/MyDrive/hotpot/dev-v2.0.json'

with open(train_path, 'r') as f:
    train_data = json.load(f)

with open(dev_path, 'r') as f:
    dev_data = json.load(f)

all_data = train_data["data"] + dev_data["data"]

print("Loaded articles:", len(all_data))

['train-v2.0.json', 'dev-v2.0.json']
Loaded articles: 477


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9.,?\'\" ]', '', text)
    return text.strip()

documents = []

for article in all_data:
    for paragraph in article["paragraphs"]:
        context = paragraph["context"]

        # preprocessing
        cleaned = clean_text(context)

        documents.append(cleaned)

print("Documents after preprocessing:", len(documents))

Documents after preprocessing: 20239


In [ ]:
from langchain_core.documents import Document


structured_docs = []

for i, doc in enumerate(documents):
    structured_docs.append(
        Document(
            page_content=doc,
            metadata={"doc_id": i}
        )
    )

print("Structured docs ready:", len(structured_docs))

Structured docs ready: 20239


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(structured_docs)

print("Total chunks:", len(chunks))

Total chunks: 42270


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(chunks, embeddings)

print("Vector DB created!")

/tmp/ipykernel_39040/1106047142.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector DB created!


In [ ]:
def retrieve_with_score(question):
    return db.similarity_search_with_score(question, k=3)